In [4]:
import librosa
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np

c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

# --------------------------
# Load CSV
# --------------------------
df = pd.read_csv('voxpopuli_with_mfcc.csv')  # replace with your CSV path

# Encode labels
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['lang']).astype(int)
num_labels = len(le.classes_)

label2id = {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))}
id2label = {int(v): str(k) for k, v in zip(le.classes_, le.transform(le.classes_))}

# Train-validation split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

In [6]:
# --------------------------
# Audio loader
# --------------------------
def load_audio(path, target_sr=16000):
    audio, _ = librosa.load(path, sr=target_sr)
    return torch.tensor(audio, dtype=torch.float)

# --------------------------
# Dataset
# --------------------------
class AudioDataset(Dataset):
    def __init__(self, df):
        self.paths = df['audio_path'].tolist()
        self.labels = df['label_id'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        audio = load_audio(self.paths[idx])
        label = self.labels[idx]
        return {"audio": audio, "label": torch.tensor(label, dtype=torch.long)}

# --------------------------
# Collate function
# --------------------------
def collate_fn(batch):
    audios = [item['audio'].tolist() for item in batch]  # list of lists
    labels = torch.stack([item['label'] for item in batch])
    return {"input_values": audios, "labels": labels}

# --------------------------
# DataLoaders
# --------------------------
train_loader = DataLoader(AudioDataset(train_df), batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(AudioDataset(val_df), batch_size=4, shuffle=False, collate_fn=collate_fn)


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
# Feature extractor
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")

# Load Wav2Vec2
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label
)

# Freeze encoder
for param in model.wav2vec2.parameters():
    param.requires_grad = False

model.to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=1e-4)


Using device: cuda


c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Pc\.cache\huggingface\hub\models--facebook--wav2vec2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\configuration_

KeyboardInterrupt: 

In [ ]:
import os

epochs = 3
checkpoint_dir = "/content/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        batch_input = feature_extractor(
            batch["input_values"],
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        ).input_values.to(device)

        labels = batch["labels"].to(device)

        outputs = model(input_values=batch_input, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Save checkpoint every 50 steps
        if (step + 1) % 50 == 0:
            ckpt_path = os.path.join(checkpoint_dir, f"epoch{epoch+1}_step{step+1}.pt")
            torch.save(model.state_dict(), ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


Saved checkpoint: /content/checkpoints/epoch1_step50.pt
Saved checkpoint: /content/checkpoints/epoch1_step100.pt
Saved checkpoint: /content/checkpoints/epoch1_step150.pt
Saved checkpoint: /content/checkpoints/epoch1_step200.pt
Saved checkpoint: /content/checkpoints/epoch1_step250.pt
Saved checkpoint: /content/checkpoints/epoch1_step300.pt
Saved checkpoint: /content/checkpoints/epoch1_step350.pt
Saved checkpoint: /content/checkpoints/epoch1_step400.pt
Saved checkpoint: /content/checkpoints/epoch1_step450.pt
Saved checkpoint: /content/checkpoints/epoch1_step500.pt
Saved checkpoint: /content/checkpoints/epoch1_step550.pt
Saved checkpoint: /content/checkpoints/epoch1_step600.pt
Saved checkpoint: /content/checkpoints/epoch1_step650.pt
Saved checkpoint: /content/checkpoints/epoch1_step700.pt
Saved checkpoint: /content/checkpoints/epoch1_step750.pt
Saved checkpoint: /content/checkpoints/epoch1_step800.pt
Saved checkpoint: /content/checkpoints/epoch1_step850.pt
Saved checkpoint: /content/check

In [ ]:
model.eval()
all_probs, all_labels, all_audio_paths = [], [], []

with torch.no_grad():
    for batch_idx, batch in enumerate(val_loader):
        batch_input = feature_extractor(
            batch["input_values"],
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        ).input_values.to(device)

        labels = batch["labels"].to(device)
        logits = model(batch_input).logits
        probs = torch.softmax(logits, dim=-1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

        batch_start = batch_idx * val_loader.batch_size
        batch_end = batch_start + len(batch["labels"])
        all_audio_paths.extend(val_df['audio_path'].iloc[batch_start:batch_end].tolist())

all_probs = np.vstack(all_probs)
all_labels = np.concatenate(all_labels)

prob_df = pd.DataFrame(all_probs, columns=[id2label[i] for i in range(num_labels)])
prob_df.insert(0, "audio_path", all_audio_paths)
prob_df.insert(1, "true_label", [id2label[i] for i in all_labels])
prob_df.to_csv("language_probabilities.csv", index=False)

print("Saved probabilities to language_probabilities.csv")


In [ ]:
# --------------------------
# Load CSV
# --------------------------
df = pd.read_csv('/content/drive/MyDrive/voxpopuli_with_mfcc.csv')

# Encode labels
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['lang']).astype(int)
num_labels = len(le.classes_)

label2id = {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))}
id2label = {int(v): str(k) for k, v in zip(le.classes_, le.transform(le.classes_))}

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])


# --------------------------
# Audio loader
# --------------------------
def load_audio(path, target_sr=16000):
    audio, _ = librosa.load(path, sr=target_sr)
    return audio  # <-- RETURN NUMPY, NOT TENSOR


# --------------------------
# Dataset
# --------------------------
class AudioDataset(Dataset):
    def __init__(self, df):
        self.paths = df['audio_path'].tolist()
        self.labels = df['label_id'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        audio = load_audio(self.paths[idx])
        label = self.labels[idx]
        return {
            "audio": audio,                     # numpy array
            "label": torch.tensor(label, dtype=torch.long)
        }


# --------------------------
# Collate function (FIXED)
# --------------------------
def collate_fn(batch):
    audios = [item["audio"] for item in batch]          # keep numpy arrays
    labels = torch.stack([item["label"] for item in batch])
    return {"input_values": audios, "labels": labels}


# --------------------------
# DataLoaders
# --------------------------
train_loader = DataLoader(AudioDataset(train_df), batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(AudioDataset(val_df), batch_size=4, shuffle=False, collate_fn=collate_fn)


# --------------------------
# Model
# --------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/wav2vec2-base"
)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label
)

# Freeze encoder
for param in model.wav2vec2.encoder.layers[-2:].parameters():
    param.requires_grad = True

model.to(device)


# --------------------------
# Optimizer
# --------------------------
optimizer = AdamW(model.parameters(), lr=1e-4)


# --------------------------
# Training Loop (FIXED)
# --------------------------
import os
epochs = 3
checkpoint_dir = "/content/drive/MyDrive/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):

        batch_inputs = feature_extractor(
            batch["input_values"],
            sampling_rate=16000,
            padding="longest",            # IMPORTANT FIX
            return_tensors="pt",
            return_attention_mask=True    # IMPORTANT FIX
        )

        input_values = batch_inputs.input_values.to(device)
        attention_mask = batch_inputs.attention_mask.to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_values=input_values,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (step + 1) % 50 == 0:
            ckpt_path = os.path.join(checkpoint_dir, f"epoch{epoch+1}_step{step+1}.pt")
            torch.save(model.state_dict(), ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / len(train_loader):.4f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# ======= Full Wav2Vec2 fine-tuning script with checkpointing and resume (Colab-ready) =======

# Install dependencies (run once)
# !pip install transformers datasets accelerate soundfile librosa --quiet

import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    Wav2Vec2ForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import librosa
import pandas as pd
from tqdm.auto import tqdm

# -------------------------
# Config
# -------------------------
CSV_PATH = "voxpopuli_with_mfcc.csv"   # your CSV
CHECKPOINT_DIR = "av2vec2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

TARGET_SR = 16000
MAX_AUDIO_SEC = 8
BATCH_SIZE = 8               
EPOCHS = 20
UNFREEZE_LAST_N = 2
BASE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
PATIENCE = 3
SAVE_EVERY_N_STEPS = 200      # checkpoint every N steps
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"  # only use AMP if GPU


print("Using device:", DEVICE)
torch.cuda.is_available()
print(torch.cuda.get_device_name())
# -------------------------
# Load CSV and prepare labels
# -------------------------
df = pd.read_csv(CSV_PATH)
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['lang'])
num_labels = len(le.classes_)
label2id = {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))}
id2label = {int(v): str(k) for k, v in zip(le.classes_, le.transform(le.classes_))}

train_df, val_df = train_test_split(df, test_size=0.12, random_state=42, stratify=df['label_id'])

# -------------------------
# Audio loader with clipping
# -------------------------
def load_audio_np(path, target_sr=TARGET_SR, max_sec=MAX_AUDIO_SEC):
    y, _ = librosa.load(path, sr=target_sr)
    max_samples = int(max_sec * target_sr)
    if len(y) > max_samples:
        y = y[:max_samples]
    return y.astype(np.float32)

# -------------------------
# Dataset + collate
# -------------------------
class AudioDataset(Dataset):
    def __init__(self, df):
        self.paths = df['audio_path'].tolist()
        self.labels = df['label_id'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        audio = load_audio_np(self.paths[idx])
        label = int(self.labels[idx])
        return {"audio": audio, "label": label}

def collate_fn(batch):
    audios = [item['audio'] for item in batch]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    return {"input_values": audios, "labels": labels}

train_loader = DataLoader(
    AudioDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2 if DEVICE.type=="cuda" else 0, pin_memory=True
)
val_loader = DataLoader(
    AudioDataset(val_df), batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2 if DEVICE.type=="cuda" else 0, pin_memory=True
)

# -------------------------
# Feature extractor + model
# -------------------------
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label
)

# Freeze most of the encoder
for p in model.wav2vec2.parameters():
    p.requires_grad = False

# Unfreeze last N layers
try:
    encoder_layers = model.wav2vec2.encoder.layers
    n_layers = len(encoder_layers)
    for i in range(n_layers - UNFREEZE_LAST_N, n_layers):
        for p in encoder_layers[i].parameters():
            p.requires_grad = True
    print(f"Unfroze last {UNFREEZE_LAST_N} encoder layers (out of {n_layers}).")
except Exception as e:
    print("Could not partially unfreeze encoder layers:", e)

# Train classification head and projector if exists
for p in model.classifier.parameters():
    p.requires_grad = True
if hasattr(model, "projector"):
    for p in model.projector.parameters():
        p.requires_grad = True

model.to(DEVICE)

# -------------------------
# Optimizer and scheduler
# -------------------------
encoder_params = []
head_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith("wav2vec2.encoder") or name.startswith("wav2vec2.feature_extractor"):
        encoder_params.append(param)
    else:
        head_params.append(param)

optim_groups = [
    {"params": encoder_params, "lr": BASE_LR},
    {"params": head_params,    "lr": HEAD_LR}
]
optimizer = AdamW(optim_groups, weight_decay=WEIGHT_DECAY)

total_steps = int(len(train_loader) * EPOCHS)
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# -------------------------
# Evaluation function
# -------------------------
def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in dataloader:
            batch_inputs = feature_extractor(
                batch["input_values"],
                sampling_rate=TARGET_SR,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            )
            input_values = batch_inputs.input_values.to(DEVICE)
            attention_mask = batch_inputs.attention_mask.to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(input_values=input_values, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels

# -------------------------
# Training loop with checkpointing and resume
# -------------------------
best_val_loss = float("inf")
best_epoch = -1
patience_counter = 0
start_epoch = 1
global_step = 0

# ✅ Automatically resume from latest checkpoint in Drive (if exists)
ckpt_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("checkpoint_step")]
if ckpt_files:
    latest_ckpt = max(ckpt_files, key=lambda x: int(x.split("step")[1].split(".")[0]))
    RESUME_FROM = os.path.join(CHECKPOINT_DIR, latest_ckpt)
    checkpoint = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint["epoch"]
    global_step = checkpoint["global_step"]
    print(f"Resuming training from epoch {start_epoch}, global_step {global_step}")

print("Starting training on device:", DEVICE)

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for step, batch in enumerate(progress):
        global_step += 1

        batch_inputs = feature_extractor(
            batch["input_values"],
            sampling_rate=TARGET_SR,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True
        )

        input_values = batch_inputs.input_values.to(DEVICE)
        attention_mask = batch_inputs.attention_mask.to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(input_values=input_values, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        progress.set_postfix(loss=(running_loss / (step + 1)))

        # --- save checkpoint every N steps ---
        if global_step % SAVE_EVERY_N_STEPS == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_step{global_step}.pt")
            torch.save({
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
            }, ckpt_path)
            print(f"\nSaved checkpoint at step {global_step}")

    # End of epoch: validate
    val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader)
    print(f"\nEpoch {epoch} summary: train_loss={(running_loss/len(train_loader)):.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"best_model_epoch{epoch}.pt"))
        print("Saved NEW best model.")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"No improvement. Patience {patience_counter}/{PATIENCE}")

    # Early stopping
    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

# -------------------------
# Final evaluation with best model
# -------------------------
print(f"Training complete. Best epoch: {best_epoch}, best_val_loss: {best_val_loss:.4f}")
if best_epoch > 0:
    best_path = os.path.join(CHECKPOINT_DIR, f"best_model_epoch{best_epoch}.pt")
    model.load_state_dict(torch.load(best_path))
    model.to(DEVICE)
    val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader)
    print("Final best model evaluation — val_loss:", val_loss, "val_acc:", val_acc)
    print(classification_report(val_labels, val_preds, target_names=list(le.classes_)))
    print("Confusion matrix:\n", confusion_matrix(val_labels, val_preds))



c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
NVIDIA GeForce RTX 3060 Ti


c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Pc\.cache\huggingface\hub\models--facebook--wav2vec2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_

Unfroze last 2 encoder layers (out of 12).


C:\Users\Pc\AppData\Local\Temp\ipykernel_17456\1735829006.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


Starting training on device: cuda


Epoch 1/20:   0%|          | 0/1650 [00:00<?, ?it/s]

In [ ]:
# ======= Full Wav2Vec2 fine-tuning script with checkpointing + detailed logs =======

import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    Wav2Vec2ForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import librosa
import pandas as pd
from tqdm.auto import tqdm
import librosa
from IPython.display import Audio


# -------------------------
# Config
# -------------------------
CSV_PATH = "voxpopuli_with_mfcc.csv"   # your CSV
CHECKPOINT_DIR = "av2vec2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

TARGET_SR = 16000
MAX_AUDIO_SEC = 8
BATCH_SIZE = 8
EPOCHS = 20
UNFREEZE_LAST_N = 2
BASE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
PATIENCE = 3
SAVE_EVERY_N_STEPS = 200
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("Using device:", DEVICE)
torch.cuda.is_available()
print(torch.cuda.get_device_name())
# -------------------------
# Load CSV
# -------------------------
df = pd.read_csv(CSV_PATH)

print("Sample audio playback:")
idx = 13268
audio_path = df.loc[idx, "audio_path"]

y, sr = librosa.load(audio_path, sr=None)
Audio(y, rate=sr)

le = LabelEncoder()
df['label_id'] = le.fit_transform(df['lang'])

num_labels = len(le.classes_)
label2id = {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))}
id2label = {int(v): str(k) for k, v in zip(le.classes_, le.transform(le.classes_))}

train_df, val_df = train_test_split(
    df, test_size=0.12, random_state=42, stratify=df['label_id']
)

# -------------------------
# Print dataset info
# -------------------------
print("\n===== DATA INFO =====")
print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches per epoch (train): {len(train_df) // BATCH_SIZE}")
print(f"Total epochs: {EPOCHS}")

# -------------------------
# Audio loader
# -------------------------
def load_audio_np(path, target_sr=TARGET_SR, max_sec=MAX_AUDIO_SEC):
    y, _ = librosa.load(path, sr=target_sr)
    max_samples = int(max_sec * target_sr)
    if len(y) > max_samples:
        y = y[:max_samples]
    return y.astype(np.float32)

# -------------------------
# Dataset + collate
# -------------------------
class AudioDataset(Dataset):
    def __init__(self, df):
        self.paths = df['audio_path'].tolist()
        self.labels = df['label_id'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        audio = load_audio_np(self.paths[idx])
        label = int(self.labels[idx])
        return {"audio": audio, "label": label}

def collate_fn(batch):
    audios = [item['audio'] for item in batch]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    return {"input_values": audios, "labels": labels}

train_loader = DataLoader(
    AudioDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2 if DEVICE.type == "cuda" else 0,
    pin_memory=True
)

val_loader = DataLoader(
    AudioDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2 if DEVICE.type == "cuda" else 0,
    pin_memory=True
)

# -------------------------
# Model + feature extractor
# -------------------------
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label
)

# Freeze encoder
for p in model.wav2vec2.parameters():
    p.requires_grad = False

# Unfreeze last N layers
try:
    encoder_layers = model.wav2vec2.encoder.layers
    n_layers = len(encoder_layers)
    for i in range(n_layers - UNFREEZE_LAST_N, n_layers):
        for p in encoder_layers[i].parameters():
            p.requires_grad = True
    print(f"\nUnfroze last {UNFREEZE_LAST_N}/{n_layers} encoder layers.")
except Exception as e:
    print("\nPartial unfreeze failed:", e)

# Always train classifier
for p in model.classifier.parameters():
    p.requires_grad = True
if hasattr(model, "projector"):
    for p in model.projector.parameters():
        p.requires_grad = True

model.to(DEVICE)

# -------------------------
# Optimizer + scheduler
# -------------------------
encoder_params = []
head_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if "wav2vec2.encoder" in name:
        encoder_params.append(param)
    else:
        head_params.append(param)

optimizer = AdamW([
    {"params": encoder_params, "lr": BASE_LR},
    {"params": head_params, "lr": HEAD_LR}
], weight_decay=WEIGHT_DECAY)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# -------------------------
# Evaluation with logs
# -------------------------
def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    progress = tqdm(dataloader, desc="🔍 Validating", leave=False)

    with torch.no_grad():
        for batch in progress:
            batch_inputs = feature_extractor(
                batch["input_values"],
                sampling_rate=TARGET_SR,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            )

            input_values = batch_inputs.input_values.to(DEVICE)
            attention_mask = batch_inputs.attention_mask.to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(
                input_values=input_values,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

            progress.set_postfix({"val_loss": f"{total_loss / (len(all_preds)+1):.4f}"})

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels

# -------------------------
# Resume from checkpoint (if any)
# -------------------------
best_val_loss = float("inf")
best_epoch = -1
patience_counter = 0
start_epoch = 1
global_step = 0

ckpt_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("checkpoint_step")]

if ckpt_files:
    latest_ckpt = max(ckpt_files, key=lambda x: int(x.split("step")[1].split(".")[0]))
    ckpt_path = os.path.join(CHECKPOINT_DIR, latest_ckpt)

    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = checkpoint["epoch"]
    global_step = checkpoint["global_step"]

    print("\n✅ Resumed from checkpoint:")
    print(f"   Epoch: {start_epoch}")
    print(f"   Global step: {global_step}")
    print(f"   Remaining epochs: {EPOCHS - start_epoch + 1}")

# -------------------------
# Training loop with detailed logs
# -------------------------
print("\n🚀 Starting training on device:", DEVICE)

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    print(f"\n================== EPOCH {epoch}/{EPOCHS} ==================")

    progress = tqdm(enumerate(train_loader), total=len(train_loader))

    for step, batch in progress:
        global_step += 1

        batch_inputs = feature_extractor(
            batch["input_values"],
            sampling_rate=TARGET_SR,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True
        )

        input_values = batch_inputs.input_values.to(DEVICE)
        attention_mask = batch_inputs.attention_mask.to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(
                input_values=input_values,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()

        avg_loss = running_loss / (step + 1)
        current_lr = scheduler.get_last_lr()[0]

        progress.set_description(f"Epoch {epoch}/{EPOCHS} | Step {step+1}/{len(train_loader)}")
        progress.set_postfix({
            "loss": f"{loss.item():.4f}",
            "avg_loss": f"{avg_loss:.4f}",
            "lr": f"{current_lr:.2e}"
        })

        # Save checkpoint
        if global_step % SAVE_EVERY_N_STEPS == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_step{global_step}.pt")
            torch.save({
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
            }, ckpt_path)
            print(f"\n💾 Saved checkpoint at step {global_step}")

    # ---- Validation ----
    val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader)

    print(f"\n📊 EPOCH {epoch} SUMMARY:")
    print(f"   Train Loss: {running_loss/len(train_loader):.4f}")
    print(f"   Val Loss:   {val_loss:.4f}")
    print(f"   Val Acc:    {val_acc:.4f}")

    # ---- Best model saving + early stopping ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"best_model_epoch{epoch}.pt"))
        print("✅ New best model saved.")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"⏳ No improvement. Patience {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print("🛑 Early stopping triggered.")
        break

# -------------------------
# Final evaluation
# -------------------------
print(f"\n🏁 Training complete! Best epoch: {best_epoch} | Best Val Loss: {best_val_loss:.4f}")

if best_epoch > 0:
    best_model_path = os.path.join(CHECKPOINT_DIR, f"best_model_epoch{best_epoch}.pt")
    model.load_state_dict(torch.load(best_model_path))
    model.to(DEVICE)

    val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader)

    print("\n✅ Final Best Model Evaluation:")
    print(f"Val Loss: {val_loss}")
    print(f"Val Acc:  {val_acc}")
    print(classification_report(val_labels, val_preds, target_names=list(le.classes_)))
    print("Confusion matrix:\n", confusion_matrix(val_labels, val_preds))

c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
NVIDIA GeForce RTX 3060 Ti
Sample audio playback:

===== DATA INFO =====
Train samples: 13200
Validation samples: 1800
Batch size: 8
Batches per epoch (train): 1650
Total epochs: 20


c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Unfroze last 2/12 encoder layers.


C:\Users\Pc\AppData\Local\Temp\ipykernel_1340\2807346365.py:193: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)



🚀 Starting training on device: cuda

================== EPOCH 1/20 ==================
